# AM5061 · Week 10 · Bagasse cogeneration

**Design of Thermal and Fluid Systems** · Applied Mechanics, IIT Madras · Jul–Nov 2026

Run the two setup cells below once, then work down the notebook. Nothing needs to be installed on your own machine.


## Setup

Run these two cells first. The second one writes the course helper module, so this notebook is self-contained.


In [ ]:
#@title Install the property library  { display-mode: "form" }
!pip install -q CoolProp openpyxl
print('CoolProp ready')


In [ ]:
%%writefile am5061.py
"""AM5061 - Design of Thermal and Fluid Systems.
Shared helpers for the course notebooks.

Design of Thermal and Fluid Systems, IIT Madras, Jul-Nov 2026.

This module is deliberately thin. It wraps CoolProp with names and units that
match the lecture notation, adds an Excel writer that produces workbooks you
can actually filter, and sets a consistent plot style. It does NOT hide the
engineering: every case notebook still writes its own equations.

Install (first cell of any Colab notebook):
    !pip install -q CoolProp openpyxl

Units are SI throughout, with ONE exception that is flagged everywhere it
appears: temperatures in function arguments named `..._C` are in Celsius,
because that is how the case briefs state them. Everything internal is kelvin.
"""
from __future__ import annotations

import math

from CoolProp.CoolProp import PropsSI, PhaseSI

__all__ = [
    "K", "C", "State", "state", "sat_liquid", "sat_vapour", "p_sat", "T_sat",
    "glide", "h_fg", "critical", "fluids", "solve", "sweep", "to_excel",
    "style_plots", "NAVY", "ORANGE", "BLUE", "MUTED",
    "lmtd", "effectiveness", "ntu_required", "exergy", "T0_REF", "P0_REF",
]

# ---------------------------------------------------------------- constants
NAVY, ORANGE, BLUE, MUTED = "#1F3864", "#ED7D31", "#4472C4", "#59626E"
T0 = 273.15


def K(t_celsius: float) -> float:
    """Celsius -> kelvin. Use this at the boundary, never inside a formula."""
    return t_celsius + T0


def C(t_kelvin: float) -> float:
    """Kelvin -> Celsius, for reporting only."""
    return t_kelvin - T0


# ------------------------------------------------------------------- states
class State:
    """A thermodynamic state. Immutable, and it knows its own fluid.

    Construct it with any two independent properties:
        State("R134a", P=1e6, T=K(70))
        State("Water", P=101325, Q=0)      # saturated liquid
        State("R134a", P=p_cond, H=h2)

    Then read properties as attributes: .T .p .h .s .d .cp .x
    Attribute names match the lecture notation, not CoolProp's letter codes,
    so a student reading the notebook does not need the CoolProp manual open.
    """

    _MAP = {"T": "T", "P": "P", "H": "H", "S": "S", "D": "D", "Q": "Q"}

    def __init__(self, fluid: str, **kw):
        if len(kw) != 2:
            raise ValueError(
                f"a state needs exactly two properties, got {list(kw)}. "
                "Two and only two - that is the phase rule, not a quirk."
            )
        (n1, v1), (n2, v2) = kw.items()
        for n in (n1, n2):
            if n not in self._MAP:
                raise ValueError(f"unknown property {n!r}; use T, P, H, S, D or Q")
        self.fluid, self._args = fluid, (n1, v1, n2, v2)

    def _get(self, what: str) -> float:
        n1, v1, n2, v2 = self._args
        return PropsSI(what, n1, v1, n2, v2, self.fluid)

    # Named so they read like the equations on the slides.
    T  = property(lambda s: s._get("T"),  doc="temperature, K")
    p  = property(lambda s: s._get("P"),  doc="pressure, Pa")
    h  = property(lambda s: s._get("H"),  doc="specific enthalpy, J/kg")
    s  = property(lambda s: s._get("S"),  doc="specific entropy, J/kg.K")
    d  = property(lambda s: s._get("D"),  doc="density, kg/m3")
    cp = property(lambda s: s._get("C"),  doc="cp, J/kg.K")
    mu = property(lambda s: s._get("V"),  doc="dynamic viscosity, Pa.s")
    k  = property(lambda s: s._get("L"),  doc="thermal conductivity, W/m.K")
    x  = property(lambda s: s._get("Q"),  doc="vapour quality, - (=-1 if single phase)")

    @property
    def T_C(self) -> float:
        return C(self.T)

    @property
    def phase(self) -> str:
        n1, v1, n2, v2 = self._args
        return PhaseSI(n1, v1, n2, v2, self.fluid)

    def __repr__(self):
        try:
            return (f"State({self.fluid}: {self.T_C:.2f} C, {self.p/1e5:.3f} bar, "
                    f"h={self.h/1e3:.2f} kJ/kg, {self.phase})")
        except Exception:
            return f"State({self.fluid}, {self._args})"


def state(fluid: str, **kw) -> State:
    """Shorthand for State(...)."""
    return State(fluid, **kw)


def sat_liquid(fluid: str, *, T=None, p=None) -> State:
    """Saturated liquid at T or p. Give one, not both."""
    if (T is None) == (p is None):
        raise ValueError("give exactly one of T or p")
    return State(fluid, T=T, Q=0) if T is not None else State(fluid, P=p, Q=0)


def sat_vapour(fluid: str, *, T=None, p=None) -> State:
    """Saturated vapour at T or p."""
    if (T is None) == (p is None):
        raise ValueError("give exactly one of T or p")
    return State(fluid, T=T, Q=1) if T is not None else State(fluid, P=p, Q=1)


def p_sat(fluid: str, T: float) -> float:
    """Saturation pressure, Pa. For a BLEND this is the bubble-point pressure."""
    return PropsSI("P", "T", T, "Q", 0, fluid)


def T_sat(fluid: str, p: float) -> float:
    """Saturation temperature, K.

    WARNING for blends: a zeotropic mixture has no single saturation
    temperature. This returns the BUBBLE point. Use glide() to see the spread.
    """
    return PropsSI("T", "P", p, "Q", 0, fluid)


def glide(fluid: str, p: float) -> float:
    """Dew minus bubble temperature at p, K. Zero for a pure fluid."""
    return (PropsSI("T", "P", p, "Q", 1, fluid)
            - PropsSI("T", "P", p, "Q", 0, fluid))


def h_fg(fluid: str, *, T=None, p=None) -> float:
    """Latent heat, J/kg."""
    return sat_vapour(fluid, T=T, p=p).h - sat_liquid(fluid, T=T, p=p).h


def critical(fluid: str) -> dict:
    """Critical point, for checking you are not extrapolating past it."""
    return {"T": PropsSI("TCRIT", fluid), "p": PropsSI("PCRIT", fluid)}


def fluids() -> list:
    """Every fluid CoolProp knows. There are about 130."""
    import CoolProp
    return sorted(CoolProp.__fluids__)


# ------------------------------------------------------------------ solvers
def solve(f, x0, *, tol=1e-10, max_iter=200, bracket=None):
    """Find x where f(x) = 0.

    Uses Brent's method when you give a bracket (robust, always converges if
    the bracket is valid), otherwise secant from x0. Raises with a readable
    message rather than returning a wrong answer silently, which is the whole
    problem with doing this in a spreadsheet.
    """
    from scipy.optimize import brentq, newton
    if bracket is not None:
        a, b = bracket
        fa, fb = f(a), f(b)
        if fa * fb > 0:
            raise ValueError(
                f"f({a:g})={fa:g} and f({b:g})={fb:g} have the same sign, so no "
                "root is bracketed. Widen the bracket or check the equation."
            )
        return brentq(f, a, b, xtol=tol, maxiter=max_iter)
    return newton(f, x0, tol=tol, maxiter=max_iter)


def sweep(fn, values, *, name="x"):
    """Run fn(v) for each v and collect the results as a list of dicts.

    fn must return a dict. The sweep variable is added under `name`, so the
    result drops straight into to_excel().
    """
    rows = []
    for v in values:
        out = fn(v)
        if not isinstance(out, dict):
            raise TypeError("the swept function must return a dict of results")
        rows.append({name: v, **out})
    return rows


# -------------------------------------------------------------------- excel
def to_excel(path, sheets: dict, *, sources=None, summary=None, title=None):
    """Write a workbook that is genuinely usable.

    Every numeric cell is written as a number (not text), AutoFilter is on,
    the header row is frozen, and columns are sized to content. A Summary and
    a Sources sheet are always present, because a result you cannot trace is
    not an engineering deliverable.

        sheets  = {"Sweep": [ {...}, {...} ], ...}   list of dicts per sheet
        sources = [ ("what", "where it came from"), ... ]
        summary = [ ("quantity", value, "units"), ... ]
    """
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

    wb = Workbook()
    wb.remove(wb.active)
    head_font = Font(bold=True, color="FFFFFF", name="Calibri")
    head_fill = PatternFill("solid", fgColor="1F3864")

    def _write(ws, rows, headers=None):
        headers = headers or (list(rows[0].keys()) if rows else [])
        for j, hname in enumerate(headers, 1):
            c = ws.cell(row=1, column=j, value=hname)
            c.font, c.fill = head_font, head_fill
            c.alignment = Alignment(horizontal="left")
        for i, row in enumerate(rows, 2):
            for j, hname in enumerate(headers, 1):
                v = row.get(hname)
                # Numbers stay numbers. This is the single most common way a
                # delivered workbook turns out not to be filterable.
                if isinstance(v, bool):
                    v = str(v)
                elif isinstance(v, (int, float)) and not isinstance(v, bool):
                    v = float(v) if isinstance(v, float) else v
                ws.cell(row=i, column=j, value=v)
        if rows:
            ws.auto_filter.ref = (f"A1:{get_column_letter(len(headers))}"
                                  f"{len(rows) + 1}")
        ws.freeze_panes = "A2"
        for j, hname in enumerate(headers, 1):
            width = max([len(str(hname))] +
                        [len(f"{r.get(hname)}") for r in rows[:200]]) + 3
            ws.column_dimensions[get_column_letter(j)].width = min(width, 42)

    # Summary first, so it is what opens.
    ws = wb.create_sheet("Summary")
    ws["A1"] = title or "AM5061 results"
    ws["A1"].font = Font(bold=True, size=14, color="1F3864")
    r = 3
    for item in (summary or []):
        for j, v in enumerate(item, 1):
            ws.cell(row=r, column=j, value=v)
        r += 1
    ws.column_dimensions["A"].width = 46
    ws.column_dimensions["B"].width = 18
    ws.column_dimensions["C"].width = 14

    for sname, rows in sheets.items():
        _write(wb.create_sheet(sname[:31]), rows)

    ws = wb.create_sheet("Sources")
    _write(ws, [{"item": a, "source": b} for a, b in (sources or [])])

    wb.save(path)
    return path


# --------------------------------------------------------------- plot style
def style_plots():
    """Match the lecture decks, so figures in a report look like the slides."""
    import matplotlib as mpl
    mpl.rcParams.update({
        "figure.figsize": (7.2, 4.4), "figure.dpi": 110,
        "axes.edgecolor": MUTED, "axes.labelcolor": NAVY,
        "axes.titlecolor": NAVY, "axes.titlesize": 11.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
        "xtick.color": MUTED, "ytick.color": MUTED,
        "font.size": 10, "legend.frameon": False,
        "axes.prop_cycle": mpl.cycler(color=[NAVY, ORANGE, BLUE, "#7F9DB9"]),
    })


# ------------------------------------------------- exchanger relations
def lmtd(dT1, dT2):
    """Log-mean temperature difference.

    Falls back to the arithmetic mean when the two ends are within 1% of each
    other, where the log form is numerically unstable and the two agree to
    better than 0.01% anyway.
    """
    dT1, dT2 = float(dT1), float(dT2)
    if dT1 <= 0 or dT2 <= 0:
        raise ValueError(
            f"temperature difference must be positive at both ends "
            f"(got {dT1:g} and {dT2:g}). A non-positive end means the streams "
            "cross, which no exchanger of this configuration can do."
        )
    if abs(dT1 - dT2) < 0.01*max(dT1, dT2):
        return 0.5*(dT1 + dT2)
    return (dT1 - dT2)/math.log(dT1/dT2)


def effectiveness(config, NTU, Cr):
    """Effectiveness for the standard configurations.

    config: 'counter', 'parallel', 'shell1'  (one shell pass, 2/4/... tube passes),
            'cross-both-unmixed' (approximate), 'cross-Cmax-mixed', 'cross-Cmin-mixed'

    Cr = C_min/C_max. Cr = 0 is the phase-change limit and every configuration
    collapses to the same expression, which is why boilers and condensers are
    easy and everything else is not.
    """
    if NTU < 0:
        raise ValueError("NTU cannot be negative")
    if not 0 <= Cr <= 1:
        raise ValueError(f"Cr must be between 0 and 1, got {Cr:g}")
    if Cr == 0:                       # phase change on one side
        return 1 - math.exp(-NTU)
    if config == "counter":
        if abs(Cr - 1) < 1e-12:
            return NTU/(1 + NTU)
        e = math.exp(-NTU*(1 - Cr))
        return (1 - e)/(1 - Cr*e)
    if config == "parallel":
        return (1 - math.exp(-NTU*(1 + Cr)))/(1 + Cr)
    if config == "shell1":
        r = math.sqrt(1 + Cr*Cr)
        e = math.exp(-NTU*r)
        return 2/(1 + Cr + r*(1 + e)/(1 - e))
    if config == "cross-both-unmixed":
        return 1 - math.exp((math.exp(-Cr*NTU**0.78) - 1)*NTU**0.22/Cr)
    if config == "cross-Cmax-mixed":
        return (1/Cr)*(1 - math.exp(-Cr*(1 - math.exp(-NTU))))
    if config == "cross-Cmin-mixed":
        return 1 - math.exp(-(1 - math.exp(-Cr*NTU))/Cr)
    raise ValueError(f"unknown configuration {config!r}")


def ntu_required(config, eps, Cr, hi=200.0):
    """Invert effectiveness() for NTU. Design direction, rather than rating."""
    eps_max = effectiveness(config, hi, Cr)
    if eps >= eps_max:
        raise ValueError(
            f"effectiveness {eps:g} is unreachable for {config} at Cr={Cr:g}; "
            f"the limit as NTU->infinity is {eps_max:.6f}. "
            "Change the configuration or accept less."
        )
    return solve(lambda n: effectiveness(config, n, Cr) - eps, 1.0,
                 bracket=(1e-9, hi))


# --------------------------------------------------------------- exergy
T0_REF, P0_REF = 303.15, 101325.0      # 30 C, sea level: the Chennai dead state


def exergy(st, T0=T0_REF, p0=P0_REF):
    """Specific flow exergy, J/kg:  (h - h0) - T0*(s - s0).

    The dead state is the ambient the plant actually sits in, so it is a
    DESIGN CHOICE, not a constant. Report which one you used - a Chennai
    dead state and a European one give different answers for the same plant.
    """
    ref = State(st.fluid, T=T0, P=p0)
    return (st.h - ref.h) - T0*(st.s - ref.s)


---
## The case

A **3500 TCD** sugar mill. Bagasse fires a boiler at **87 bar / 515 °C**; a
**backpressure turbine** exhausts to the **3.5 bar** process header. How much
surplus power can the mill export, and what happens off-season?

Deliverable **D-10**: steam balance and export power at **100% and 60% crush
rate**, with the turbine off-design handled by the **Stodola ellipse law**.

### The thing that makes cogeneration different

In a condensing plant you choose the steam flow to suit the power demand. In a
backpressure cogeneration plant the **process decides the steam flow**, and the
power is whatever falls out. Power is a by-product. Get that backwards and every
number afterwards is wrong.


## 1. The mill, on a mass basis

In [ ]:
import am5061 as am
import numpy as np, matplotlib.pyplot as plt
from CoolProp.CoolProp import PropsSI
from scipy.optimize import brentq
am.style_plots()

TCD          = 3500.0          # tonnes cane per day
HOURS        = 24.0
bagasse_pct  = 30.0            # % on cane
steam_pct    = 50.0            # % on cane, boiler output
process_pct  = 40.0            # % on cane, demand at 3.5 bar
mill_kWh_t   = 25.0            # kWh per tonne cane, internal electrical demand

p_hp, T_hp = 87e5, am.K(515)   # boiler outlet
p_bp       = 3.5e5             # process header
eta_s_t    = 0.80              # turbine isentropic efficiency
eta_gen    = 0.96

def mill(crush=1.0):
    cane   = TCD*crush/HOURS                    # t/h
    bag    = cane*bagasse_pct/100
    steam  = cane*steam_pct/100                 # t/h
    proc   = cane*process_pct/100
    return {"crush, %": crush*100, "cane, t/h": cane, "bagasse, t/h": bag,
            "steam raised, t/h": steam, "process demand, t/h": proc,
            "internal power, kW": cane*mill_kWh_t}

for c in (1.0, 0.6):
    m = mill(c)
    print("  " + "  ".join(f"{k} {v:.2f}" for k, v in m.items()))


## 2. The turbine at design point

In [ ]:
st_in  = am.State("Water", P=p_hp, T=T_hp)
h_in, s_in = st_in.h, st_in.s
h_out_s = PropsSI("H", "P", p_bp, "S", s_in, "Water")
h_out   = h_in - eta_s_t*(h_in - h_out_s)
st_out  = am.State("Water", P=p_bp, H=h_out)

print(f"  inlet   {p_hp/1e5:.0f} bar, {am.C(T_hp):.0f} C, h = {h_in/1e3:.1f} kJ/kg")
print(f"  ideal exhaust  h_s = {h_out_s/1e3:.1f} kJ/kg")
print(f"  real  exhaust  h   = {h_out/1e3:.1f} kJ/kg, {st_out.T_C:.1f} C, x = {st_out.x:.4f}")
print(f"  specific work  {(h_in-h_out)/1e3:.2f} kJ/kg")
if 0 <= st_out.x <= 1:
    print(f"  NOTE exhaust is wet at x = {st_out.x:.3f}. Below about 0.88 the last")
    print("  stage erodes. Check this before you accept a backpressure.")
else:
    print("  exhaust is superheated, so no erosion concern.")


## 3. Steam balance and export power

The turbine passes the steam the boiler raises. Process takes what it needs. Any
**excess exhaust has nowhere to go** — it is vented, and that is pure loss.


In [ ]:
def balance(crush=1.0, steam_pct=steam_pct, process_pct=process_pct):
    m = mill(crush)
    m["steam raised, t/h"] = m["cane, t/h"]*steam_pct/100
    m["process demand, t/h"] = m["cane, t/h"]*process_pct/100
    m_s   = m["steam raised, t/h"]*1000/3600            # kg/s through the turbine
    P_gen = m_s*(h_in - h_out)*eta_gen                  # W
    vent  = max(m["steam raised, t/h"] - m["process demand, t/h"], 0.0)
    short = max(m["process demand, t/h"] - m["steam raised, t/h"], 0.0)
    export = P_gen - m["internal power, kW"]*1e3
    return {**m, "turbine flow, kg/s": m_s,
            "power generated, kW": P_gen/1e3,
            "export, kW": export/1e3,
            "exhaust vented, t/h": vent, "process shortfall, t/h": short,
            "heat-to-power ratio": (m["process demand, t/h"]*1000/3600
                                    *(h_out - PropsSI("H","P",p_bp,"Q",0,"Water")))/max(P_gen,1)}

print(f"{'':22s}{'100% crush':>14}{'60% crush':>14}")
b100, b60 = balance(1.0), balance(0.6)
for k in ("cane, t/h","steam raised, t/h","process demand, t/h","turbine flow, kg/s",
          "power generated, kW","internal power, kW","export, kW","exhaust vented, t/h"):
    print(f"{k:22s}{b100[k]:14.2f}{b60[k]:14.2f}")
print(f"\n  export falls {100*(1-b60['export, kW']/b100['export, kW']):.1f}% "
      f"for a 40% cut in crush - roughly proportional, because everything scales"
      "\n  with cane and the turbine still sees its design pressure ratio.")


## 4. Off-design: the Stodola ellipse

At part load the turbine passes less steam, so its **inlet pressure falls**.
Stodola's law says the swallowing capacity follows an ellipse:

`ṁ / ṁ_d = √[ (p_in² − p_out²) / (p_in,d² − p_out,d²) ] · √(T_d / T_in)`

Solve it for the inlet pressure the machine will actually sit at.


In [ ]:
m_design = b100["turbine flow, kg/s"]

def stodola_inlet_pressure(m_dot, T_in=T_hp, p_out=p_bp,
                           m_d=m_design, p_in_d=p_hp, T_d=T_hp):
    """Invert the ellipse for p_in at a given flow."""
    def f(p_in):
        ratio = np.sqrt(max(p_in**2 - p_out**2, 1e-9)
                        /(p_in_d**2 - p_out**2))*np.sqrt(T_d/T_in)
        return ratio*m_d - m_dot
    return brentq(f, p_out*1.001, p_in_d*1.5)

def turbine_offdesign(crush):
    b   = balance(crush)
    m_s = b["turbine flow, kg/s"]
    p_in = stodola_inlet_pressure(m_s)
    st_i = am.State("Water", P=p_in, T=T_hp)
    h_o_s = PropsSI("H","P",p_bp,"S",st_i.s,"Water")
    h_o   = st_i.h - eta_s_t*(st_i.h - h_o_s)
    P     = m_s*(st_i.h - h_o)*eta_gen
    return {"crush, %": crush*100, "flow, kg/s": m_s, "p_in, bar": p_in/1e5,
            "specific work, kJ/kg": (st_i.h - h_o)/1e3,
            "power, kW": P/1e3,
            "export, kW": (P - b["internal power, kW"]*1e3)/1e3,
            "x_exhaust": am.State("Water", P=p_bp, H=h_o).x}

print(f"{'crush %':>9}{'flow kg/s':>11}{'p_in bar':>10}{'w kJ/kg':>10}"
      f"{'power kW':>11}{'export kW':>11}")
rows = []
for c in np.arange(0.40, 1.05, 0.05):
    r_ = turbine_offdesign(float(c)); rows.append(r_)
    print(f"{r_['crush, %']:9.0f}{r_['flow, kg/s']:11.3f}{r_['p_in, bar']:10.2f}"
          f"{r_['specific work, kJ/kg']:10.2f}{r_['power, kW']:11.1f}{r_['export, kW']:11.1f}")


**Read that middle column.** The boiler cannot hold 87 bar at part
load: the turbine simply will not swallow the steam at that pressure. Inlet
pressure falls, the pressure ratio falls with it, and the **specific work drops**.
So power falls faster than flow does. That non-linearity is the whole reason
off-design analysis exists.


In [ ]:
fig, (a1,a2) = plt.subplots(1,2, figsize=(11.8,4.2))
cr = [r_["crush, %"] for r_ in rows]
a1.plot(cr, [r_["p_in, bar"] for r_ in rows], "o-", lw=2.4, color=am.NAVY)
a1.set_xlabel("crush rate  (%)"); a1.set_ylabel("turbine inlet pressure  (bar)")
a1.set_title("Stodola: the machine sets its own inlet pressure")
a2.plot(cr, [r_["power, kW"] for r_ in rows], "o-", lw=2.4, color=am.ORANGE, label="generated")
a2.plot(cr, [r_["export, kW"] for r_ in rows], "o-", lw=2.4, color=am.BLUE, label="exported")
a2.axhline(0, color=am.MUTED, lw=1)
a2.set_xlabel("crush rate  (%)"); a2.set_ylabel("power  (kW)")
a2.set_title("Both fall with crush; export falls slightly faster"); a2.legend(fontsize=9)
plt.tight_layout(); plt.show()

naive = balance(0.6)["power generated, kW"]
stod  = turbine_offdesign(0.6)["power, kW"]
print(f"\n  At 60% crush:")
print(f"    scaling the design point linearly gives {naive:8.1f} kW")
print(f"    Stodola, with p_in falling to 52 bar,   {stod:8.1f} kW")
print(f"    the naive method overstates power by    {100*(naive-stod)/stod:8.1f}%")
print("    That is the entire reason off-design analysis exists.\n")

zero = [r_ for r_ in rows if r_["export, kW"] <= 0]
if zero:
    print(f"  Export reaches zero at about {max(z['crush, %'] for z in zero):.0f}% crush.")
    print("  Below that the mill imports. The internal load is nearly fixed while")
    print("  generation falls with cane, so the crossover comes sooner than expected.")
else:
    print("  The mill exports across the whole range studied.")


## 5. The deliverable

In [ ]:
bal_rows = [balance(float(c)) for c in np.arange(0.4, 1.05, 0.1)]
path = am.to_excel("AM5061_D10_Cogeneration.xlsx",
    {"Steam balance": bal_rows, "Turbine off-design (Stodola)": rows},
    title="AM5061 D-10 . 3500 TCD bagasse cogeneration",
    summary=[("Cane crush", TCD, "TCD"),
             ("Boiler steam", f"{p_hp/1e5:.0f} bar / {am.C(T_hp):.0f} C", ""),
             ("Process header", p_bp/1e5, "bar"),
             ("Turbine isentropic efficiency", eta_s_t, "-"),
             ("Specific work at design", (h_in-h_out)/1e3, "kJ/kg"),
             ("Power generated, 100% crush", b100["power generated, kW"], "kW"),
             ("Export, 100% crush", b100["export, kW"], "kW"),
             ("Export, 60% crush", turbine_offdesign(0.6)["export, kW"], "kW"),
             ("Exhaust quality at design", st_out.x, "-")],
    sources=[("Water/steam", "CoolProp, IAPWS-95"),
             ("Off-design turbine", "Stodola ellipse law"),
             ("Mill ratios", "bagasse 30% on cane, steam 50%, process 40%, "
                             "25 kWh/t internal - TYPICAL VALUES, confirm for the site"),
             ("Case data", "AM5061 brief D-10")])
print("written:", path)


## What to hand in

1. The steam balance at 100% and 60% crush.
2. Export power at both, **using Stodola for the off-design point** — not by
   scaling the design-point number linearly.
3. The crush rate at which export reaches zero, and what the mill should do
   about it.
4. The exhaust quality, and whether it is acceptable.
5. The workbook.

**State your assumptions.** The mill ratios here (30% bagasse on cane, 50%
steam, 40% process, 25 kWh/t) are typical Indian values, not measurements. Every
number downstream scales with them, so a report that does not declare them is
not checkable.
